In [ ]:
!pip install -q transformers bitsandbytes peft accelerate datasets

import warnings
warnings.filterwarnings("ignore")

categories = [
    "ORDER", "SHIPPING", "CANCEL", "INVOICE",
    "PAYMENT", "REFUND", "FEEDBACK", "CONTACT",
    "ACCOUNT", "DELIVERY", "SUBSCRIPTION"
]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_and_split_dataset(dataset_path, dataset_size=None):
    df = pd.read_csv(dataset_path)

    if dataset_size is None:
        df_small = df
    else:
        df_small, _ = train_test_split(
            df, train_size=dataset_size, stratify=df['category'], random_state=42
        )

    df_train, df_temp = train_test_split(
        df_small, test_size=0.2, stratify=df_small['category'], random_state=42
    )

    df_val, df_test = train_test_split(
        df_temp, test_size=0.5, stratify=None, random_state=42
    )

    return df_train, df_val, df_test

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

def model_loader(model_id):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map={"": 0},
            quantization_config=bnb_config,
            torch_dtype=torch.float16
        )
        
        device = torch.device("cuda")

    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
        )
        
        device = torch.device("cpu")

    generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

    return model, tokenizer, generator, device

In [ ]:
def general_inference_tinyllama(df_test, generator, categories):
    correct = 0
    total = len(df_test)
    predictions = []

    system_prompt = {
        "role": "system",
        "content": (
            "You are an AI assistant that classifies user support queries into ONE and ONLY ONE of the following categories:\n\n"
            f"{chr(10).join(f'- {c}' for c in categories)}\n\n"
            "INSTRUCTIONS:\n"
            "- Read the user's query carefully.\n"
            "- Select the ONE category from the list that BEST matches the query.\n"
            "- ONLY return the category name, EXACTLY as it appears in the list.\n"
            "- DO NOT invent new categories."
            "- DO NOT include any extra words, explanations, punctuation, or formatting.\n\n"
        )
    }

    for step, (_, row) in enumerate(df_test.iterrows(), 1):
        user_prompt = {
            "role": "user",
            "content": row["instruction"]
        }

        assistant_prompt = {
            "role": "assistant",
            "content": "Labeled Category:"
        }

        messages = [system_prompt, user_prompt, assistant_prompt]
        prompt = "\n".join(f"<|{msg['role']}|>\n{msg['content']}" for msg in messages)

        output = generator(prompt, max_new_tokens=10)[0]["generated_text"]

        predicted_category = "UNKNOWN"
        for cat in categories:
            if cat in output:
                predicted_category = cat
                break

        actual_category = row["category"]
        is_correct = predicted_category == actual_category
        correct += is_correct
        predictions.append({
            "instruction": row["instruction"],
            "actual": actual_category,
            "predicted": predicted_category,
            "correct": is_correct
        })

    accuracy = correct / total * 100

    return accuracy

In [ ]:
import re
from datasets import Dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def format_chat(example, tokenizer):
    example["text"] = f"Instruction: {example['instruction']}\nLabeled Category: {example['category']}{tokenizer.eos_token}"
    return example

def tokenize_function(example, tokenizer, max_length=512):
    tokenized = tokenizer(example["text"], padding="max_length", truncation=True, max_length=max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

def finetune_category_tinyllama(df_train, model, tokenizer, device):
    dataset = Dataset.from_pandas(df_train[["instruction", "category"]])
    dataset = dataset.map(lambda x: format_chat(x, tokenizer))
    
    dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir="./tinyllama-lora-category-classifier",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=3,
        logging_steps=100,
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        label_names=["labels"]
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    trainer.train()
    return model

def predict_output(text, model, tokenizer, device):
    prompt = f"Instruction: {text}\nLabeled Category:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output

def extract_category(text, categories):
    match = re.search(r"Labeled Category:\s*(.*)", text, re.IGNORECASE)
    if match:
        label_section = match.group(1).strip().upper()

        categories_upper = [cat.upper() for cat in categories]

        for cat in sorted(categories_upper, key=len, reverse=True):
            if cat in label_section:
                return cat

    return "NOT FOUND"

def evaluate_model_tinyllama(df_test, model, tokenizer, device, categories):
    predictions = []
    correct = 0
    total = len(df_test)

    for _, row in df_test.iterrows():
        instruction = row["instruction"]
        true_category = row["category"].strip().upper()

        pred = predict_output(instruction, model, tokenizer, device)
        pred_category = extract_category(pred, categories)
        pred_category = pred_category.strip().upper()
        predictions.append(pred)

        if pred_category == true_category:
            correct += 1

    accuracy = (correct / total) * 100

    return accuracy

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

dataset_path =  'path_to_dataset'
model_id = "Doctor-Shotgun/TinyLlama-1.1B-32k-Instruct"
dataset_size = 5000

df_train, df_val, df_test = load_and_split_dataset(dataset_path, dataset_size)

model_initial, tokenizer_initial, generator, device = model_loader(model_id)

accuracy = general_inference_tinyllama(df_test, generator, categories)
print(f"Accuracy without Fine-tuning [TinyLlama]: {accuracy:.2f}%")

model_category_tinyllama = finetune_category_tinyllama(df_train, model_initial, tokenizer_initial, device)
accuracy = evaluate_model_tinyllama(df_test, model_category_tinyllama, tokenizer_initial, device, categories)
print(f"Accuracy with LoRa Fine-tuning [TinyLlama]: {accuracy:.2f}%")